# 02 — TFT prototype

We over-fit a *single* ticker on a *short* window. This validates the
model architecture; generalisation comes later.

In [1]:
import numpy as np
import torch
import torch.nn as nn
import yfinance as yf
from torch.utils.data import DataLoader, TensorDataset

from backend.perception.temporal.tft_model import TemporalFusionTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"

In [2]:
df = yf.download(
    "SPY",
    start="2023-01-01",
    end="2024-01-01",
    interval="1d",
    auto_adjust=False,
    progress=False,
)
data = df[["Open", "High", "Low", "Close", "Volume"]].to_numpy(dtype=np.float32)
print("Days:", len(data))

Days: 250


## Build sliding windows

WINDOW = 60 days, HORIZON = 5 days return target

In [3]:
WINDOW = 60
HORIZON = 5

# Per-window normalisation
xs, ys = [], []
for i in range(len(data) - WINDOW - HORIZON):
    x = data[i : i + WINDOW].copy()
    x_mean, x_std = x.mean(0, keepdims=True), x.std(0, keepdims=True) + 1e-6
    x_norm = (x - x_mean) / x_std
    target_return = (data[i + WINDOW + HORIZON - 1, 3] - data[i + WINDOW - 1, 3]) / data[
        i + WINDOW - 1, 3
    ]
    xs.append(x_norm)
    ys.append(target_return)
xs, ys = torch.from_numpy(np.stack(xs)), torch.tensor(ys, dtype=torch.float32)
print("X:", xs.shape, "y:", ys.shape)

ds = TensorDataset(xs, ys)
loader = DataLoader(ds, batch_size=8, shuffle=True)

X: torch.Size([185, 60, 5]) y: torch.Size([185])


## Training loop

In [4]:
# NB: the production TFT defaults to OHLCV_WINDOW_MINUTES (240); we
# override to 60 for this short notebook prototype.
model = TemporalFusionTransformer(
    num_features=5,
    hidden_dim=64,
    num_heads=4,
    lstm_layers=1,
    output_dim=64,
    window_length=WINDOW,
    dropout=0.1,
).to(device)
pred_head = nn.Linear(64, 1).to(device)
opt = torch.optim.AdamW(list(model.parameters()) + list(pred_head.parameters()), lr=3e-4)

losses = []
for epoch in range(60):
    epoch_loss = 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        emb, _ = model(x)
        pred = pred_head(emb).squeeze(-1)
        loss = ((pred - y) ** 2).mean()
        opt.zero_grad()
        loss.backward()
        opt.step()
        epoch_loss += loss.item() * x.size(0)
    losses.append(epoch_loss / len(ds))
    if epoch % 5 == 0:
        print(f"epoch {epoch:3d}  mse = {losses[-1]:.6f}")

epoch   0  mse = 0.035522


epoch   5  mse = 0.002465


epoch  10  mse = 0.001402


epoch  15  mse = 0.000750


epoch  20  mse = 0.000666


...

epoch  35  mse = 0.000504


epoch  40  mse = 0.000394


epoch  45  mse = 0.000374


epoch  50  mse = 0.000389


epoch  55  mse = 0.000377


## Sanity assertion

We expect over-fitting MSE < 1e-3 within 60 epochs.

In [5]:
final = losses[-1]
print(f"Final training MSE: {final:.6f}")
assert final < 1e-3, f"TFT failed over-fit sanity check: {final:.6f}"
print("PASS — TFT architecture works.")

Final training MSE: 0.000330
PASS — TFT architecture works.
